# SatDiff — Kaggle training

Kaggle gives 30 GPU-hours/week, guaranteed. Colab guarantees nothing — that is
why this notebook exists.

## One-time setup

1. **Phone-verify the account** — kaggle.com/settings. GPU *and* Internet are both
   locked behind it.
2. Right sidebar: **Accelerator → GPU T4 x2**. Not P100 — the T4 has fp16 tensor
   cores and this config trains in mixed precision.
3. Right sidebar: **Internet → On**.
4. **Add-ons → Secrets** → add `HF_TOKEN`, a *write* token from
   hf.co/settings/tokens.

## The thing that bites

`/kaggle/working` is wiped between sessions and there is no Drive to fall back
on. The Hub is the only durable store, so `HF_TOKEN` is not optional — without
it a dead session costs you the whole run.

Every cell sets `PYTHONPATH` and `cd`s for itself. Changing the accelerator
restarts the kernel and drops that state, and a cell that assumed an earlier
cell had set it fails with a confusing `No module named 'satdiff'`.

In [ ]:
# 1. GPU check. Anything other than True here and nothing below will work.
import torch
print("cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "Sidebar -> Accelerator -> GPU T4 x2")

In [ ]:
# 2. Code, deps, HF auth. The repo is public, so no token is needed to clone.
import os, sys

REPO = '/kaggle/working/satdiff-v2'

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception as e:
    print(f'WARNING: no HF_TOKEN ({e}). Training still runs, but checkpoints')
    print('will not be pushed anywhere durable — a dead session loses the run.')

%cd /kaggle/working
if not os.path.isdir(REPO):
    !git clone -q https://github.com/DrKingSchultz69/satdiff-v2.git
%cd $REPO
!git pull -q

!pip install -q -r requirements.txt

os.environ['PYTHONPATH'] = f'{REPO}/src'
sys.path.insert(0, f'{REPO}/src')
print('ready')

In [ ]:
# 3. Data — 94 MB, ~2 min. Splits are assigned by SHA-256 of each file path,
# not listdir() order, so this reproduces the exact same train/val/test split
# as every other machine. That is what makes resuming here sound.
%cd /kaggle/working/satdiff-v2
!python scripts/download_data.py
!python scripts/make_splits.py

In [ ]:
# 4. Train. --resume pulls last.pt from the Hub when it is not on disk, so a
# killed session costs at most `hub_push_every` epochs.
#
# CHECK THE FIRST LINES. 'resumed from epoch N' means it worked. 'starting
# fresh' means the Hub had nothing and you are about to redo every GPU-hour
# already spent — stop it and fix the checkpoint before letting it run.
%cd /kaggle/working/satdiff-v2
!PYTHONPATH=src python -m satdiff.train --config configs/v1.yaml --resume

In [ ]:
# 5. The eye test. One row per class, same 4 seeds every time.
# Four identical images in a row is mode collapse, whatever KID says.
import glob
from IPython.display import Image, display

grids = sorted(glob.glob('/kaggle/working/satdiff-v2/results/grids/*.png'))
if grids:
    print(grids[-1])
    display(Image(grids[-1]))
else:
    print('none yet — the first grid lands at epoch 5')

In [ ]:
# 6. Eval: KID + CAS. Trains a ResNet-18 on real data first (~10 min), then
# scores 2,700 generated images. ~30 min total.
#
# Bars, fixed before training started (docs/eval-plan.md):
#   KID  ship <0.05   good <0.02
#   CAS  ship >=65%   good >=80%
%cd /kaggle/working/satdiff-v2
!PYTHONPATH=src python -m satdiff.eval --config configs/v1.yaml --split val

In [ ]:
# 7. Every eval run so far, newest last.
import os
import pandas as pd

csv = '/kaggle/working/satdiff-v2/results/experiments.csv'
display(pd.read_csv(csv)) if os.path.exists(csv) else print('no eval runs yet')